# 12.07 · Observability and operations: logs, metrics, traces, SLOs and incidents

**Module 12 · Production & Deployment** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 3–4 h |
| 📋 Prerequisites | Lessons 12.01 (SLOs), 12.04–12.06 (the service), 10.04 (tracing). |
| 🔑 Keys | none (optional: `LANGSMITH_API_KEY` for the 🌐 trace lookup; Docker for the 🐳 Prometheus drill) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Choose** the right signal for a question: logs (what happened to *this* request?), metrics (how is the service doing *overall*?), traces (what did the agent do *inside* one run?).
2. **Correlate** them with ids: a request id on every response and log line, and the same ids in each trace's metadata.
3. **Write** structured JSON logs that carry ids, timings and costs, and never message text or credentials.
4. **Expose** Prometheus metrics with low-cardinality labels, and read latency percentiles from histograms.
5. **Turn SLOs into alerts** (multi-window burn rate) and check that one fires during an injected outage.
6. **Run an incident** with a runbook: detect, mitigate, find the cause, write a blameless postmortem.

## 📚 Official reading (verified 2026-10-16)

| Page | Read it for |
|---|---|
| [Prometheus Python client: FastAPI](https://prometheus.github.io/client_python/exporting/http/fastapi-gunicorn/) | Serving metrics from an ASGI app (and the `add_route` caveat on 0.26.0) |
| [Prometheus: metric and label naming](https://prometheus.io/docs/practices/naming/) | `_total`, `_seconds`, base units, label cardinality |
| [Prometheus: histograms and summaries](https://prometheus.io/docs/practices/histograms/) | `histogram_quantile`, bucket choice, error of the estimate |
| [Prometheus: alerting rules](https://prometheus.io/docs/prometheus/latest/configuration/alerting_rules/) | `expr`, `for`, labels, annotations, templating |
| [Google SRE workbook: alerting on SLOs](https://sre.google/workbook/alerting-on-slos/) | Burn rate, the 14.4× one-hour page, multi-window alerts |
| [LangSmith: trace query syntax](https://docs.langchain.com/langsmith/trace-query-syntax) | `and(eq(metadata_key, …), eq(metadata_value, …))` filters |
| [LangSmith: add metadata and tags](https://docs.langchain.com/langsmith/add-metadata-tags) | Metadata in the run config reaches every trace |
| [OpenTelemetry Python: getting started](https://opentelemetry.io/docs/languages/python/getting-started/) | Tracer providers, spans, attributes, exporters |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "langgraph-checkpoint-sqlite>=3.1.1,<4" "fastapi>=0.141.1,<0.142" "uvicorn>=0.54.0,<1" "pydantic-settings>=2.16.0,<3" "httpx>=0.28.1,<1" "pyjwt[crypto]>=2.15.1,<3" "langgraph-checkpoint-postgres>=3.2.0,<4" "psycopg[binary]>=3.3.6,<4" "prometheus-client>=0.26.0,<1" "opentelemetry-sdk>=1.42.1,<2" "langsmith>=0.14.11,<1"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("prometheus-client", "opentelemetry-sdk", "langsmith", "fastapi",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/12-production-and-deployment/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm12-observability'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m12-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 09 LIVE FAQ index): no usage telemetry.
# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# fastembed (Project 09 LIVE FAQ index) imports tqdm.auto; without ipywidgets it warns once. Progress bars still work.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

---
## Part 1 · Three signals, three questions

| Signal | Question | Our service | Cost to keep |
|---|---|---|---|
| **Logs** | What happened to request `abc-123`? | One JSON line per request and per run (`mastery.service.observability`) | Grows with traffic; keep days to weeks |
| **Metrics** | Is the error rate up? Is p99 TTFT over 3 s? | `GET /metrics` in Prometheus format | Fixed per series; keep months |
| **Traces** | Which node or tool made this run slow or wrong? | LangSmith (or OpenTelemetry), with the service's ids in metadata | Per run; sample if large |

They meet at **ids**. Every response carries `X-Request-ID`; the log lines of that request and of the run it started carry it; the run's trace has it in its metadata, together with `service_run_id`, `thread_id`, `user_id`, `release` and `mode`.

---
## Part 2 · Request ids and structured logs

We start the service with `log_format="json"` and `log_level="INFO"` (what the container uses) and collect its log lines in a list instead of the terminal, so we can look at them.

In [ ]:
# 🧪 Offline: one request with our own X-Request-ID, then read the log lines it produced.
import json
import logging

import httpx

from mastery.service.app import create_app, serve_in_background
from mastery.service.client import bearer, stream_events
from mastery.service.desk import support_manager
from mastery.service.observability import JsonFormatter
from mastery.service.settings import ServiceSettings

class Collect(logging.Handler):
    """Keep each log record as the JSON object the container would print."""
    def __init__(self):
        super().__init__()
        self.lines = []
    def emit(self, record):
        self.lines.append(json.loads(JsonFormatter().format(record)))

class Service:
    """Start the app in a background thread; its logs go to `self.logs.lines`."""
    def __init__(self, build=support_manager, **settings):
        self.app = create_app(build, ServiceSettings(_env_file=None, log_format="json", log_level="INFO", **settings))
        self.logs = Collect()
        logging.getLogger("mastery.service").handlers = [self.logs]     # instead of stderr
    def __enter__(self):
        self.url, self.stop = serve_in_background(self.app)
        self.client = httpx.Client(base_url=self.url, timeout=30)
        return self.client
    def __exit__(self, *exc):
        self.client.close()
        self.stop()

QUESTION = "How do I rotate an API key? Call me on PRIVATE-555-0100."
service = Service(release="1.4.0")
with service as c:
    tid = c.post("/threads", headers=bearer("demo-customer")).json()["thread_id"]
    r = c.post(f"/threads/{tid}/runs", json={"message": QUESTION}, headers=bearer("demo-customer") | {"X-Request-ID": "support-ticket-4711"})
    print("response X-Request-ID:", r.headers["x-request-id"])
    run_id = r.json()["run_id"]
    stream_events(c, run_id, "demo-customer")
for line in service.logs.lines:
    if line.get("request_id") == "support-ticket-4711":
        print(json.dumps(line))
print("\nmessage text in any log line:", any("PRIVATE" in json.dumps(l) for l in service.logs.lines))

**You should now see** `response X-Request-ID: support-ticket-4711`, two JSON lines carrying that id (`"event": "http.request"` for `POST /threads/{thread_id}/runs` with status 202, and `"event": "run.finished"` with the `run_id`, `ttft_s`, `duration_s`, `tokens` and `cost_usd`), and `message text in any log line: False`.

### What just happened?

The request id travelled from the HTTP request into the background run: `RunManager.submit` reads it from a context variable, which `asyncio.create_task` copies into the run's task. When a customer quotes the id from an error screen, one search finds the request, the run, and (Part 5) its trace.

The log lines hold ids, timings, sizes and costs only. The message, which here contains a phone number, never reaches the logs. Logs are copied to many places and kept for weeks; keep user text where you can delete it (Lesson 12.05).

---
## Part 3 · Metrics

We run 40 FAQ questions through a desk that **fails 8% of its model calls on purpose** (`fail_rate`, a simulated provider outage; each run makes several model calls) with 0.05 s of simulated model latency, then scrape `GET /metrics` like Prometheus would.

In [ ]:
# 🧪 Offline: traffic with injected failures, then a scrape.
import time

from prometheus_client.parser import text_string_to_metric_families

flaky_desk = lambda s: support_manager(s, delay_s=0.05, fail_rate=0.08, seed=7)  # noqa: E731
service = Service(build=flaky_desk, max_concurrent=8)
with service as c:
    run_ids = []
    for i in range(40):
        tid = c.post("/threads", headers=bearer("demo-customer")).json()["thread_id"]
        run_ids.append(c.post(f"/threads/{tid}/runs", json={"message": "How do I rotate an API key?"},
                              headers=bearer("demo-customer")).json()["run_id"])
    for run_id in run_ids:
        stream_events(c, run_id, "demo-customer")
    scrape = c.get("/metrics").text
    records = service.app.state.manager.records()

families = {f.name: f for f in text_string_to_metric_families(scrape)}
runs_by_status = {s.labels["status"]: int(s.value) for s in families["agent_runs"].samples if s.name.endswith("_total")}
print("runs by status:", runs_by_status)
requests = [{"route": s.labels["route"], "status": s.labels["status"], "count": int(s.value)}
            for s in families["agent_http_requests"].samples if s.name.endswith("_total")]
table(sorted(requests, key=lambda r: -r["count"]), ["route", "status", "count"])
print("\nseries for user ids or run ids:", sum(run_ids[0] in line for line in scrape.splitlines()))

**You should now see** roughly three quarters of the runs `success` and the rest `error`, a request table with route **templates** (`/runs/{run_id}/stream`, not 40 different paths), and `series for user ids or run ids: 0`.

### What just happened?

Each label value creates a time series that Prometheus stores for months. Routes as templates and statuses as a handful of values keep the series count fixed; a `user_id` or `run_id` label would add series without limit (a "cardinality explosion"). Per-user questions belong in logs, which are made for them.

In [ ]:
# 🧪 Offline: TTFT percentiles from the histogram (what Prometheus can do) vs. exact (what we measured).
from mastery.service.slo import percentile

buckets = [(float(s.labels["le"]), s.value) for s in families["agent_run_ttft_seconds"].samples
           if s.name.endswith("_bucket")]

def bucket_quantile(q, buckets):
    """histogram_quantile: find the bucket holding the q-th observation, interpolate linearly inside it."""
    total = buckets[-1][1]
    rank, lower, below = q * total, 0.0, 0.0
    for upper, count in buckets:
        if count >= rank:
            if upper == float("inf"):
                return lower
            return lower + (upper - lower) * (rank - below) / max(count - below, 1e-9)
        lower, below = upper, count
    return lower

exact = [r.ttft_s for r in records if r.ttft_s is not None]
table([{"quantile": q, "from buckets": round(bucket_quantile(q / 100, buckets), 3), "exact": round(percentile(exact, q), 3)}
       for q in (50, 95, 99)], ["quantile", "from buckets", "exact"])

**You should now see** three rows: the bucket estimate is close to the exact value at p50 and p95 and further off at p99, where the few slowest runs sit inside a wide bucket (0.5 to 1 s) and linear interpolation can only guess.

### What just happened?

A histogram stores counts per bucket, not the values, so a percentile is an estimate whose error is the bucket width. Put bucket boundaries **at your SLO thresholds** (we have 3 s in the TTFT buckets and 60 s in the duration ones): then "what fraction is under 3 s?" is exact, even if "what is p99?" isn't.

---
## Part 4 · From SLOs to alerts

Lesson 12.01 wrote the SLOs; `apps/m12_service/ops/alerts.yml` turns them into Prometheus rules. The availability page uses the SRE workbook's **multi-window burn rate**: fire when the error ratio is above 14.4 × the budget (1% × 14.4 = 14.4%) over **both** the last hour (it's real) and the last 5 minutes (it's still happening).

In [ ]:
# 🧪 Offline: would the page fire on this traffic? Same arithmetic as the PromQL rule.
import yaml

from mastery.service.slo import burn_rate, default_slos, error_budget_policy, evaluate_slo

rules = yaml.safe_load((REPO_ROOT / "apps" / "m12_service" / "ops" / "alerts.yml").read_text(encoding="utf-8"))
fast_burn = next(r for g in rules["groups"] for r in g["rules"] if r["alert"] == "AgentErrorBudgetFastBurn")
print(fast_burn["expr"])

availability = default_slos()[0]
result = evaluate_slo(availability, records)
rate = burn_rate(availability, records, since=min(r.started_at for r in records))
print(f"availability   : {result.good}/{result.eligible} good (target {result.target:.0%})")
print(f"burn rate      : {rate:.1f}x  (page at 14.4x)")
print("page fires     :", rate > 14.4)
print("budget policy  :", error_budget_policy(result))

**You should now see** the PromQL expression, availability far under 99%, a burn rate well above 14.4×, `page fires: True`, and the `freeze` policy.

### What just happened?

A burn rate of 20× would spend a 30-day error budget in a day and a half: that wakes someone up. A slow burn (say 2× for a day) gets a ticket instead (exercise 2). Alerting on burn rate rather than "any error" keeps pages rare and meaningful: a few failed runs on a quiet night don't page, an outage does.

Every alert links to a section of [`RUNBOOK.md`](../../apps/m12_service/RUNBOOK.md), and a unit test checks that each alert in `alerts.yml` has one.

---
## Part 5 · Traces carry the same ids

`RunManager.config_for(run)` builds each run's LangGraph config: thread, step limit, a run name, a tag, and **metadata** with the service's ids. LangChain hands the config's metadata to every callback, which is how LangSmith receives it. We check it offline with a callback that records the root run's metadata.

In [ ]:
# 🧪 Offline: what metadata does the agent's root run receive?
from langchain_core.callbacks import BaseCallbackHandler

class RootMetadata(BaseCallbackHandler):
    def __init__(self):
        self.seen = []
    def on_chain_start(self, serialized, inputs, *, run_id, parent_run_id=None, tags=None, metadata=None, **kwargs):
        if parent_run_id is None:
            self.seen.append({"tags": tags, "metadata": {k: v for k, v in (metadata or {}).items() if not k.startswith("langgraph_")}})

probe = RootMetadata()
service = Service(release="1.4.0")
with service as c:
    manager = service.app.state.manager
    plain = manager.config_for
    manager.config_for = lambda run: plain(run) | {"callbacks": [probe]}
    tid = c.post("/threads", headers=bearer("demo-customer")).json()["thread_id"]
    r = c.post(f"/threads/{tid}/runs", json={"message": "How do I rotate an API key?"},
               headers=bearer("demo-customer") | {"X-Request-ID": "support-ticket-4712"})
    stream_events(c, r.json()["run_id"], "demo-customer")
print(json.dumps(probe.seen[0], indent=2))

**You should now see** `"tags": ["agent-service"]` and metadata with `release: 1.4.0`, `mode: demo`, `service_run_id`, `thread_id`, `user_id: C-1002` and `request_id: support-ticket-4712` (plus `ls_integration`, which LangGraph adds itself).

### What just happened?

With tracing on, LangSmith stores that metadata on the trace, so `and(eq(metadata_key, "request_id"), eq(metadata_value, "support-ticket-4712"))` finds it (next cell). `user_id` is a customer id here; if yours is an email, hash it first: traces are another copy of user data (Lesson 10.02 masks PII).

The same listener mechanism feeds other backends. Below, an OpenTelemetry span per finished run, exported in memory; point the exporter at an OTLP endpoint (LangSmith accepts OTel too, Lesson 10.04) and you get service-level spans next to the agent's.

In [ ]:
# 🧪 Offline: one OpenTelemetry span per run, from a RunManager listener.
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.trace import Status, StatusCode

exporter = InMemorySpanExporter()
provider = TracerProvider()                      # a local provider: nothing global changes
provider.add_span_processor(SimpleSpanProcessor(exporter))
tracer = provider.get_tracer("agent-service")

def span_for(run):
    end_ns = time.time_ns()
    start_ns = end_ns - int((run.ended_at - run.created_at) * 1e9)
    span = tracer.start_span("agent.run", start_time=start_ns, attributes={
        "service.run_id": run.run_id, "service.request_id": run.request_id or "", "run.status": run.status,
        "run.ttft_s": run.public()["ttft_s"] or -1, "gen_ai.usage.input_tokens": run.usage.get("input_tokens", 0),
        "gen_ai.usage.output_tokens": run.usage.get("output_tokens", 0)})
    if run.status in {"error", "timeout"}:
        span.set_status(Status(StatusCode.ERROR, run.error))
    span.end(end_time=end_ns)

service = Service(build=flaky_desk)
with service as c:
    service.app.state.manager.listeners.append(span_for)
    for i in range(6):
        tid = c.post("/threads", headers=bearer("demo-customer")).json()["thread_id"]
        run_id = c.post(f"/threads/{tid}/runs", json={"message": "How do I rotate an API key?"}, headers=bearer("demo-customer")).json()["run_id"]
        stream_events(c, run_id, "demo-customer")
spans = exporter.get_finished_spans()
table([{"span": s.name, "status": s.status.status_code.name, "ms": round((s.end_time - s.start_time) / 1e6, 1),
        "input_tokens": s.attributes["gen_ai.usage.input_tokens"]} for s in spans], ["span", "status", "ms", "input_tokens"])

**You should now see** six `agent.run` spans, most `UNSET` (OpenTelemetry's default for "no error") and any failed run `ERROR`, each with its duration and input tokens.

### 🌐 Find a run's trace by request id

This cell turns LangSmith tracing on for one DEMO run (the offline model: no model tokens), waits for the trace to arrive, and looks it up by the request id.

> 🌐 Talks to LangSmith (free Developer plan is enough; no model tokens). Without `LANGSMITH_API_KEY` it prints `⏭️ Skipped`.

In [ ]:
# 🌐 LangSmith: trace one DEMO run and find it by metadata (no model tokens).
if HAVE_LANGSMITH_KEY:
    from langchain_core.tracers.langchain import wait_for_all_tracers
    from langsmith import Client

    previous = os.environ.get("LANGSMITH_TRACING")
    os.environ["LANGSMITH_TRACING"] = "true"
    try:
        service = Service(release="1.4.0")
        with service as c:
            tid = c.post("/threads", headers=bearer("demo-customer")).json()["thread_id"]
            request_id = f"lesson-12-07-{int(time.time())}"
            run_id = c.post(f"/threads/{tid}/runs", json={"message": "How do I rotate an API key?"},
                            headers=bearer("demo-customer") | {"X-Request-ID": request_id}).json()["run_id"]
            stream_events(c, run_id, "demo-customer")
        wait_for_all_tracers()
    finally:
        if previous is None:
            os.environ.pop("LANGSMITH_TRACING", None)
        else:
            os.environ["LANGSMITH_TRACING"] = previous
    query = f'and(eq(metadata_key, "request_id"), eq(metadata_value, "{request_id}"))'
    found = []
    for attempt in range(15):                    # ingestion takes a few seconds
        found = list(Client().list_runs(project_name=os.environ["LANGSMITH_PROJECT"], filter=query, is_root=True, limit=5))
        if found:
            break
        time.sleep(2)
    print("filter :", query)
    if found:
        trace = found[0]
        print("found  :", trace.name, "·", trace.status, "·", trace.extra["metadata"]["service_run_id"] == run_id)
    else:
        print("not found yet: open the project in LangSmith and paste the filter above")
else:
    print("⏭️ Skipped: LANGSMITH_API_KEY is not set (see the setup cell).")

**You should now see** the filter and `found  : service-run · success · True` (the root run is named `service-run` by `config_for`).

---
## Part 6 · 🐳 Prometheus and an incident drill

`compose.yaml` has an `obs` profile that adds Prometheus (`prom/prometheus:v3.15.0`) with `ops/prometheus.yml` and `ops/alerts.yml`. The drill: start the stack with `DEMO_FAIL_RATE=0.1` (10% of model calls fail), send traffic, watch `AgentErrorBudgetFastBurn` fire, "fix" it, watch it clear. In a terminal (Git Bash; in PowerShell set `$env:DEMO_FAIL_RATE='0.1'` first):

```bash
DEMO_FAIL_RATE=0.1 docker compose -f apps/m12_service/compose.yaml --profile obs up --build -d
```

Then open http://localhost:9090/alerts. The cells do the same.

In [ ]:
# 🐳 Is Docker here and running? (no tokens)
import shutil
import subprocess

def docker(*args, timeout=600, env=None):
    result = subprocess.run(["docker", *args], cwd=REPO_ROOT, capture_output=True, text=True, timeout=timeout,
                            env=None if env is None else {**os.environ, **env})
    if result.returncode != 0:
        raise RuntimeError(f"docker {' '.join(args)} failed:\n{result.stdout}{result.stderr}")
    return (result.stdout + result.stderr).strip()

HAVE_DOCKER = False
if shutil.which("docker"):
    try:
        print("Docker server:", docker("version", "--format", "{{.Server.Version}}", timeout=20))
        HAVE_DOCKER = True
    except (RuntimeError, subprocess.TimeoutExpired):
        print("⏭️ Docker is installed but not running: start Docker Desktop, then re-run this cell.")
else:
    print("⏭️ Skipped: no docker command (install Docker Desktop for the 🐳 cells).")
COMPOSE = ["compose", "-f", "apps/m12_service/compose.yaml", "--profile", "obs"]
API, PROM = "http://127.0.0.1:8000", "http://127.0.0.1:9090"

In [ ]:
# 🐳 The drill: break it, send traffic, wait for the page (about 1-2 minutes; no tokens).
def wait_for(check, seconds, every=3):
    deadline = time.monotonic() + seconds
    while time.monotonic() < deadline:
        try:
            value = check()
            if value:
                return value
        except httpx.HTTPError:
            pass
        time.sleep(every)
    return None

def firing():
    alerts = httpx.get(f"{PROM}/api/v1/alerts", timeout=5).json()["data"]["alerts"]
    return [a["labels"]["alertname"] for a in alerts if a["state"] == "firing"]

def traffic(n):
    with httpx.Client(base_url=API, timeout=30) as c:
        for _ in range(n):
            tid = c.post("/threads", headers=bearer("demo-customer")).json()["thread_id"]
            run_id = c.post(f"/threads/{tid}/runs", json={"message": "How do I rotate an API key?"}, headers=bearer("demo-customer")).json()["run_id"]
            stream_events(c, run_id, "demo-customer")

if HAVE_DOCKER:
    docker(*COMPOSE, "up", "--build", "-d", timeout=1800, env={"DEMO_FAIL_RATE": "0.1"})
    wait_for(lambda: httpx.get(f"{API}/ready", timeout=2).status_code == 200, 90)
    started = time.monotonic()
    traffic(40)
    alerts = wait_for(lambda: "AgentErrorBudgetFastBurn" in firing() and firing(), 120)
    print(f"firing after {time.monotonic() - started:.0f} s:", alerts)
    logs = docker("compose", "-f", "apps/m12_service/compose.yaml", "logs", "--no-log-prefix", "api")
    errors = [json.loads(line) for line in logs.splitlines() if line.startswith("{") and '"run_status": "error"' in line]
    print("error log lines:", len(errors), "· first:", errors[0]["error"] if errors else "-")
else:
    print("⏭️ Skipped: Docker is not available.")

**You should now see** `firing after … s: ['AgentErrorBudgetFastBurn']` within a couple of minutes (scrapes and rule evaluations run every 15 s), and error log lines whose `error` is `SimulatedOutage: simulated provider outage (HTTP 503)`: the runbook's step 1 finds the cause in one search.

In [ ]:
# 🐳 Mitigate (the "provider" recovers), confirm, and stop. The page clears when the 5-minute window is clean.
if HAVE_DOCKER:
    docker(*COMPOSE, "up", "-d", "api", timeout=300, env={"DEMO_FAIL_RATE": "0"})
    wait_for(lambda: httpx.get(f"{API}/ready", timeout=2).status_code == 200, 90)
    traffic(20)
    query = 'sum(rate(agent_runs_total{status="error"}[1m]))'
    print("errors per second now:", wait_for(lambda: httpx.get(f"{PROM}/api/v1/query", params={"query": query}, timeout=5)
                                              .json()["data"]["result"] or ["none"], 60))
    print("Watch http://localhost:9090/alerts clear over the next 5 minutes, then run:")
    print("  docker compose -f apps/m12_service/compose.yaml --profile obs down")
else:
    print("⏭️ Skipped: Docker is not available.")

**You should now see** an error rate of 0 (or `none`) after the restart: the API container came back with a fresh process, so its counters restarted too (Prometheus's `rate` handles counter resets).

---
## Part 7 · Runbook and postmortem

An alert is only useful if the person woken by it knows what to do. [`RUNBOOK.md`](../../apps/m12_service/RUNBOOK.md) has one section per alert: the signal to read first, the likely causes, the mitigation. After the incident, a **blameless postmortem** records what happened and what changes. We write one for the Part 3 "outage" from its measured numbers.

In [ ]:
# 🧪 Offline: the runbook section for the page, and a postmortem from the Part 3 numbers.
runbook = (REPO_ROOT / "apps" / "m12_service" / "RUNBOOK.md").read_text(encoding="utf-8")
section = runbook.split("## AgentErrorBudgetFastBurn")[1].split("\n## ")[0]
print("## AgentErrorBudgetFastBurn" + section)

failed = [r for r in records if r.status == "error"]
postmortem = f"""# Drill: provider errors on the support desk (Lesson 12.07)
- Impact: {len(failed)} of {len(records)} runs failed ({len(failed) / len(records):.0%}); customers saw an error event and could resend.
- Detection: AgentErrorBudgetFastBurn (burn rate {rate:.1f}x, threshold 14.4x).
- Root cause: injected SimulatedOutage on 8% of model calls; several model calls per run made about a quarter of runs fail.
- What went well: every failed run ended with a terminal event; logs named the cause; no user text in logs.
- What didn't: no automatic retry for provider 503s.
- Actions: add ModelRetryMiddleware for transient errors (Lesson 12.02) and re-run this drill (owner: you, this week).
"""
OUT = REPO_ROOT / "data" / "outputs" / "m12"
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "postmortem-drill.md").write_text("\n".join(line.strip() for line in postmortem.splitlines()) + "\n", encoding="utf-8")
print("wrote", (OUT / "postmortem-drill.md").relative_to(REPO_ROOT))

**You should now see** the runbook's five steps for the page and `wrote data/outputs/m12/postmortem-drill.md`.

### What just happened?

The postmortem's last line is the point: an action that makes this class of incident rarer (retries on transient errors) and a check that it worked (re-running the drill). Blameless means the document asks *why the system allowed it*, not *who did it*, so people report problems early.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Prometheus memory keeps growing | A label with unbounded values (user, run, raw path) | Templates and small enums only; ids go in logs |
| `GET /metrics` returns 307 | `app.mount("/metrics", …)` is a prefix mount | Our own route with `generate_latest`, or request `/metrics/` |
| The alert never fires although runs fail | No `error` series yet (counters appear on first increment), or `for:` longer than the test | Check the series in the Prometheus UI; shorter `for` while testing |
| Every page is a false alarm | Alerting on single errors or on a short window only | Multi-window burn rate; tickets for slow burns |
| One failed run at night pages you | Low traffic: 1 failure in 10 runs is a 10× burn | Synthetic traffic (a probe), client retries, or a longer window (the workbook's low-traffic section) |
| Log search finds the request but no run | Different id names in different places | One `request_id` everywhere: header, logs, trace metadata |
| Secrets or user text in logs | Logging whole requests or exceptions with arguments | An allowlist of fields (`FIELDS`); never log bodies |
| Traces show no service ids | Metadata set somewhere the graph doesn't read | Put it in the run config (`config_for`), not in a global |

## ⚠️ Production notes

> ⚠️ **Production note: protect `/metrics`.** It reveals traffic and spend. Keep it on the internal network (our Compose file doesn't publish it beyond the stack), or set `MASTERY_SERVICE_METRICS_TOKEN` and the matching `authorization` in `prometheus.yml`.

> ⚠️ **Production note: free hosted backends.** You don't have to run Prometheus yourself: Grafana Cloud's free tier accepts Prometheus remote-write and logs, and LangSmith's Developer plan keeps traces. Check retention and limits before you rely on them.

> ⚠️ **Production note: alerts need a human.** Route `severity: page` to a phone (Alertmanager, Grafana OnCall, PagerDuty's free tier) and `ticket` to an issue tracker. An alert that goes to a channel nobody watches is a log line.

## 🏋️ Exercises

1. **(Easy)** Add an `agent_rejected_runs_total{reason}` counter for 429s (queue full, over budget) and 503s (draining). Why is `reason` a safe label?
2. **(Medium)** Write the workbook's slow-burn ticket rule: error ratio above 3 × the budget over 24 hours **and** 2 hours. How much of a 30-day budget has been spent when it fires?
3. **(Medium)** Add the release to the metrics as an `agent_build_info{release="…"} 1` gauge, and write the PromQL that compares the error ratio of two releases during a canary (Lesson 12.09).
4. **(Stretch)** Send the Part 5 spans to a real backend: an OTLP exporter to LangSmith (Lesson 10.04's settings) or to a local Jaeger container. Which attributes would you add for a support agent?

<details>
<summary><b>Show solutions</b></summary>

**1.** In `ServiceMetrics`: `self.rejected = Counter(f"{p}_rejected_runs", "Runs refused", ["reason"], registry=r)`; in `start_run`'s `except` branches: `metrics.rejected.labels("queue_full" | "over_budget" | "draining").inc()`. The label has three possible values, so it adds three series at most.

**2.**
```yaml
- alert: AgentErrorBudgetSlowBurn
  expr: |
    (sum(rate(agent_runs_total{status=~"error|timeout"}[24h])) / sum(rate(agent_runs_total[24h])) > (3 * 0.01))
    and
    (sum(rate(agent_runs_total{status=~"error|timeout"}[2h])) / sum(rate(agent_runs_total[2h])) > (3 * 0.01))
  labels: {severity: ticket}
```
At 3× for 24 hours it has spent 3 × 24 h / 720 h = 10% of the 30-day budget when it fires: worth a ticket, not a page.

**3.** `Gauge(f"{p}_build_info", "Build", ["release"], registry=r).labels(settings.release).set(1)`. Error ratio per release needs the release on the run counter (`agent_runs_total{release=…}`): one series per release is fine because releases are few. Then `sum by (release) (rate(agent_runs_total{status="error"}[10m])) / sum by (release) (rate(agent_runs_total[10m]))`.

**4.** Use `OTLPSpanExporter` with LangSmith's OTel endpoint and headers (Lesson 10.04), or `docker run -p 16686:16686 -p 4318:4318 jaegertracing/jaeger` and `endpoint="http://localhost:4318/v1/traces"`. Useful attributes: the agent that answered (`langgraph_node`), tool names called, whether an approval was requested, the customer's plan, and the model id.
</details>

## 🔎 Docs drill (5 min)

In the SRE workbook's chapter on alerting on SLOs, which burn rate and window does the recommended fast-burn page use, and how much of a 30-day error budget has been spent when it fires?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Alerting on SLOs](https://sre.google/workbook/alerting-on-slos/) and find the table of recommended burn rates.
2. The page uses a burn rate of **14.4** over **1 hour** (with a 5-minute short window), which spends **2%** of a 30-day budget.
3. Compare with `apps/m12_service/ops/alerts.yml` and `mastery.service.slo.burn_rate`'s docstring.

</details>

## ✅ Checkpoint

- [ ] I can say which signal answers which question, and how ids connect them.
- [ ] One request id appears on the response, in the request and run log lines, and in the trace metadata.
- [ ] Log lines carry ids, timings and costs, and no user text or credentials.
- [ ] Metrics use route templates and small label sets; I know why histogram percentiles are estimates.
- [ ] I can read the burn-rate rule and compute whether it fires on a set of runs.
- [ ] Each alert has a runbook section, and I wrote a postmortem with an action.
- [ ] (🌐) I found a run's trace by its request id. (🐳) The drill's page fired and I watched it clear.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Check [prometheus-client on PyPI](https://pypi.org/project/prometheus-client/) and log in [`docs/research-log.md`](../../docs/research-log.md): the current version (2026-10-16: 0.26.0; the docs say `add_route` with `make_asgi_app` needs a newer release), and whether OpenTelemetry's GenAI semantic conventions (`gen_ai.*`) are still marked experimental.

## Summary

- Logs for one request, metrics for the whole service, traces for one run's insides; ids tie them together.
- JSON logs with an allowlist of fields; no message text, no secrets.
- Prometheus metrics with low-cardinality labels; histogram buckets at your SLO thresholds.
- Multi-window burn-rate alerts from SLOs; every alert has a runbook section.
- Practise incidents: inject a failure, follow the runbook, write a blameless postmortem with actions.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 12-08-optional-langsmith-deployment.ipynb** → [`12-08-optional-langsmith-deployment.ipynb`](12-08-optional-langsmith-deployment.ipynb).

[Module 12 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)